# densepoly: training di Colab

Notebook ini hanya memanggil skrip di repo (`docs/COLAB.md`). Semua logika ada di `src/`.

Sebelum mulai:
1. Runtime → Change runtime type → **GPU (T4)**.
2. Di Google Drive harus ada `MyDrive/densepoly/data_tar/crowdai_train_dedup.tar` dan `crowdai_train_dedup.json`.
   Tunggu sampai unggahan tar selesai.
3. Isi parameter di sel berikut, lalu Runtime → Run all.

`--resume` membuat sesi berikutnya melanjutkan otomatis dari `last.pt` di Drive, jadi cukup jalankan ulang notebook ini
kalau runtime terputus. Isi `TASK` memilih pekerjaannya: `benchmark` atau `train`.

In [ ]:
# Parameters of this session
TASK = "benchmark"          # "benchmark" or "train"
DATA = "crowdai"
EXP = "b4"                  # experiment to train (configs/exp/)
SEED = 0
RUN = ""                    # named run such as "pilot_r5"; empty means seed<k>
EPOCHS = 0                  # E, the same for every run (docs/EXPERIMENTS.md §5)
MAX_HOURS = 3.5             # stop cleanly before the session ends; last.pt stays on Drive
OVERRIDES = ""              # extra config overrides, e.g. "labels.contact_radius_px=5"
BENCHMARK_EXP = "full_p2"   # the slowest model
BENCHMARK_STEPS = 200

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
!test -d /content/densepoly || git clone -q https://github.com/mahathirmuh/building-footprint-extraction.git /content/densepoly
%cd /content/densepoly
!git pull -q && git log -1 --format="commit %H, %s"

In [ ]:
# torch stays the one Colab provides (docs/COLAB.md §2)
!pip install -q -e .
!python -c "import torch, segmentation_models_pytorch as smp, numpy; print('torch', torch.__version__, 'smp', smp.__version__, 'numpy', numpy.__version__, 'cuda', torch.cuda.is_available())"

In [ ]:
import os

%env DENSEPOLY_DATA=/content/data
%env DENSEPOLY_OUT=/content/drive/MyDrive/densepoly/outputs
TAR = "/content/tar/crowdai_train_dedup.tar"
!mkdir -p /content/tar /content/data
!test -f {TAR} || cp /content/drive/MyDrive/densepoly/data_tar/crowdai_train_dedup.* /content/tar/
# A failing `!` does not stop Run all, so the assert does; a bad copy is removed, and the next run copies it again
!python scripts/pack_colab.py --data {DATA} --verify --out {TAR} || rm -f {TAR}
assert os.path.exists(TAR), "No good tar, nothing was trained: see the line above, fix it, then Run all again"
!test -d /content/data/crowdai || tar -xf {TAR} -C /content/data
!du -sh /content/data

In [ ]:
run_option = f"--run {RUN}" if RUN else ""
if TASK == "benchmark":
    !python scripts/train.py --data {DATA} --exp {BENCHMARK_EXP} --benchmark {BENCHMARK_STEPS}
elif TASK == "train":
    !python scripts/train.py --data {DATA} --exp {EXP} --seed {SEED} {run_option} --resume --max-hours {MAX_HOURS} train.epochs={EPOCHS} {OVERRIDES}
else:
    raise ValueError(TASK)

In [ ]:
# Ends the session in good order: everything written to Drive is flushed
drive.flush_and_unmount()